## 1. Install and Import Project Dependencies

Run the installation cell only when the environment does not already have the required packages and the repository installed.

In [ ]:
# Kaggle setup example.
# In a Kaggle Notebook, the repository is often uploaded as a Dataset or cloned into /kaggle/working.

from pathlib import Path

REPO_DIR = Path("/kaggle/working/arxiv-rag")

# Uncomment if you already attached the repo as a Kaggle Dataset or want to clone it manually.
# %cd /kaggle/working
# !git clone https://github.com/your-user/arxiv-rag.git
# %cd {REPO_DIR}
# %pip install -U pip
# %pip install pandas numpy pyarrow tqdm scikit-learn rank_bm25 kaggle sentence-transformers faiss-cpu torch flask
# %pip install -e .

In [ ]:
# Colab setup example.
# Replace REPO_URL with your repository URL before running.

from pathlib import Path

REPO_URL = "https://github.com/your-user/arxiv-rag.git"
REPO_DIR = Path("/content/arxiv-rag")

# Uncomment in Google Colab.
# !git clone {REPO_URL} {REPO_DIR}
# %cd {REPO_DIR}
# %pip install -U pip
# %pip install pandas numpy pyarrow tqdm scikit-learn rank_bm25 kaggle sentence-transformers faiss-cpu torch flask
# %pip install -e .

## Environment Setup

Use one of the setup cells below before running the rest of the notebook. They are intended for Colab or Kaggle and cover cloning the repository, installing dependencies, and choosing where the processed corpus will come from.

In [ ]:
# Uncomment when running in a fresh Colab or Kaggle environment.
# %pip install pandas numpy pyarrow tqdm scikit-learn rank_bm25 kaggle sentence-transformers faiss-cpu torch flask
# %pip install -e .

import gc
import json
from pathlib import Path

import pandas as pd

from arxiv_rag.evaluation import Evaluator
from cloud_eval_runner import run_cloud_evaluation
from evaluate_models import (
    _build_retriever,
    load_benchmark,
    reconcile_benchmark_with_corpus,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

In [ ]:
# Option B: build processed parquet files inside the notebook runtime from the Kaggle arXiv dataset.
# This requires Kaggle credentials in Colab, but works natively in Kaggle Notebook runtimes.

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

# Uncomment to prepare data inside the notebook.
# from arxiv_rag.dataset.prepare_data import prepare_data
# metadata_path, processed_dir = prepare_data(
#     raw_dir=RAW_DIR,
#     processed_dir=PROCESSED_DIR,
#     skip_download=False,
#     force_download=False,
#     force_process=False,
#     chunksize=100_000,
# )
# print(metadata_path)
# print(processed_dir)

In [ ]:
# Option A: use an already prepared processed corpus.
# Adjust these paths for Colab Drive or Kaggle Datasets.

CANDIDATE_DATA_FOLDERS = [
    Path("data/processed"),
    Path("/kaggle/input/arxiv-rag-processed/data/processed"),
    Path("/content/drive/MyDrive/arxiv-rag/data/processed"),
]

existing_data_folders = [path for path in CANDIDATE_DATA_FOLDERS if path.exists()]
existing_data_folders

## Data Source Setup

The repository does not contain `data/processed/` because `data/` is ignored by git. Choose one of the following approaches before loading the corpus:

1. Attach a Kaggle Dataset or mount Google Drive that already contains `part_*.parquet`.
2. Run `prepare_data(...)` inside the notebook and let it download the raw metadata from Kaggle, then build `data/processed/` locally in the notebook runtime.

For Kaggle Notebooks, the most practical setup is usually to attach a separate Dataset with prepared parquet chunks and point `data_folder` to that mounted path.

In [ ]:
import torch

candidate_model_keys = [
    "bm25",
    "tfidf",
    "minilm",
    "bge",
    "specter1",
    "specter2",
    "hybrid-weighted",
    "cross-encoder",
]

runtime_info = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_current_device": torch.cuda.current_device() if torch.cuda.is_available() else None,
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

runtime_info_df = pd.DataFrame([runtime_info])
display(runtime_info_df)

device_rows = []
for model_key in candidate_model_keys:
    model_label, factory = _build_retriever(model_key)
    retriever = factory()

    if hasattr(retriever, "device"):
        effective_device = str(retriever.device)
    elif hasattr(retriever, "dense") and hasattr(retriever.dense, "device"):
        effective_device = str(retriever.dense.device)
    elif hasattr(retriever, "base_retriever") and hasattr(retriever.base_retriever, "dense"):
        effective_device = str(retriever.base_retriever.dense.device)
    else:
        effective_device = "cpu-only or not exposed"

    device_rows.append(
        {
            "model_key": model_key,
            "model_label": model_label,
            "effective_device": effective_device,
        }
    )
    del retriever

gc.collect()
pd.DataFrame(device_rows)

## CUDA Diagnostics

This section only prints runtime diagnostics in the notebook. It does not change exported results, CSV, JSON, or LaTeX artifacts. Run it after the environment setup section so the installed runtime and repository state are already finalized.

## 2. Load Processed arXiv Corpus

This section reads processed `part_*.parquet` chunks from `data/processed` and keeps the fields used by the evaluator.

In [ ]:
data_folder = next(iter(existing_data_folders), Path("data/processed"))
part_paths = sorted(data_folder.glob("part_*.parquet"))
if not part_paths:
    raise FileNotFoundError(
        "No parquet chunks found. Either attach a prepared processed dataset or run prepare_data(...) in the setup section. "
        f"Checked folder: {data_folder}"
    )

corpus_df = pd.concat(
    [pd.read_parquet(path, columns=["id", "title", "abstract"]) for path in part_paths],
    ignore_index=True,
)
corpus_df = corpus_df.fillna({"id": "", "title": "", "abstract": ""})
print(f"Using processed corpus from: {data_folder}")
corpus_df.head()

## 3. Inspect Benchmark TSV Schema

Validate that the benchmark contains the expected columns and parse `relevant_ids` into Python lists.

In [ ]:
benchmark_path = Path("eval/benchmark.tsv")
benchmark_df = pd.read_csv(benchmark_path, sep="\t")
required_columns = {"query", "relevant_ids"}
missing_columns = required_columns - set(benchmark_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

benchmark_df["relevant_ids"] = benchmark_df["relevant_ids"].apply(json.loads)
benchmark_df.head()

## 4. Prepare Text Fields for Retrieval

The evaluator uses `title + [SEP] + abstract`. This cell mirrors that behavior and keeps a normalized id list in parallel.

In [ ]:
def normalize_arxiv_id(value):
    doc_id = str(value).strip()
    if doc_id.lower().startswith("arxiv:"):
        doc_id = doc_id[6:].strip()
    if "/" in doc_id:
        prefix, suffix = doc_id.split("/", 1)
        doc_id = f"{prefix.lower()}/{suffix}"
    return doc_id

corpus_df["id"] = corpus_df["id"].map(normalize_arxiv_id)
corpus_df["retrieval_text"] = (
    corpus_df["title"].fillna("").astype(str)
    + " [SEP] "
    + corpus_df["abstract"].fillna("").astype(str)
).str.strip()

doc_ids = corpus_df["id"].tolist()
texts = corpus_df["retrieval_text"].tolist()
corpus_df[["id", "retrieval_text"]].head()

## 5. Instantiate Retriever Implementations

Start with lightweight baselines, then swap in any registered model through the same factory used by the CLI.

In [ ]:
model_keys = ["bm25", "tfidf"]
retriever_specs = []
for model_key in model_keys:
    model_label, factory = _build_retriever(model_key)
    retriever_specs.append((model_key, model_label, factory))

retriever_specs

## 6. Build Corpus Indexes

Fit each retriever on the full prepared corpus so it can be reused across benchmark queries.

In [ ]:
fitted_retrievers = {}
for model_key, model_label, factory in retriever_specs:
    retriever = factory()
    retriever.fit(texts)
    fitted_retrievers[model_key] = {"label": model_label, "retriever": retriever}

list(fitted_retrievers)

## 7. Run Top-k Retrieval for Sample Queries

Inspect a few sample ranked lists before running the full benchmark.

In [ ]:
sample_queries = [
    "graph neural networks for molecules",
    "retrieval augmented generation",
]

preview_rows = []
for query in sample_queries:
    indices = fitted_retrievers["bm25"]["retriever"].topk(query, 5)
    for rank, index in enumerate(indices, start=1):
        preview_rows.append(
            {
                "query": query,
                "rank": rank,
                "doc_index": index,
                "id": doc_ids[index],
                "title": corpus_df.iloc[index]["title"],
            }
        )

pd.DataFrame(preview_rows)

## 8. Normalize arXiv IDs and Match Relevance

Build helper logic that compares retrieved ids with benchmark relevance labels.

In [ ]:
def binary_hits(predicted_ids, relevant_ids):
    relevant = {normalize_arxiv_id(item) for item in relevant_ids}
    normalized_predictions = [normalize_arxiv_id(item) for item in predicted_ids]
    return [1 if doc_id in relevant else 0 for doc_id in normalized_predictions]

example_row = benchmark_df.iloc[0]
example_indices = fitted_retrievers["bm25"]["retriever"].topk(example_row["query"], 5)
example_ids = [doc_ids[index] for index in example_indices]
{
    "query": example_row["query"],
    "relevant_ids": example_row["relevant_ids"],
    "predicted_ids": example_ids,
    "hits": binary_hits(example_ids, example_row["relevant_ids"]),
}

## 9. Compute Recall@k, MRR, and nDCG@k

Use the project evaluator for correctness, and keep helper code simple inside the notebook.

In [ ]:
benchmark_records = load_benchmark(benchmark_path)
filtered_benchmark, benchmark_summary = reconcile_benchmark_with_corpus(
    benchmark_records,
    set(doc_ids),
)

evaluator = Evaluator(
    retriever=fitted_retrievers["bm25"]["retriever"],
    doc_ids=doc_ids,
    texts=texts,
)
metrics = evaluator.evaluate(filtered_benchmark, k=10, show_progress=True)
metrics

## 10. Evaluate Multiple Models on the Benchmark

Run several model configurations and collect structured outputs per model.

In [ ]:
single_benchmark_results = []
for model_key, fitted in fitted_retrievers.items():
    evaluator = Evaluator(
        retriever=fitted["retriever"],
        doc_ids=doc_ids,
        texts=texts,
    )
    result = evaluator.evaluate(filtered_benchmark, k=10, show_progress=True, progress_desc=model_key)
    single_benchmark_results.append(
        {
            "model_key": model_key,
            "model_label": fitted["label"],
            "mrr": result["mrr"],
            "recall@10": result["recall@10"],
            "ndcg@10": result["ndcg@10"],
        }
    )

pd.DataFrame(single_benchmark_results).sort_values("mrr", ascending=False)

## 11. Aggregate Evaluated and Penalized Metrics

Match the project behavior by distinguishing valid evaluated queries from skipped ones.

In [ ]:
query_factor = (
    benchmark_summary["evaluated_queries"] / benchmark_summary["total_queries"]
    if benchmark_summary["total_queries"]
    else 0.0
)

aggregated_df = pd.DataFrame(single_benchmark_results)
aggregated_df["penalized_mrr"] = aggregated_df["mrr"] * query_factor
aggregated_df["penalized_recall@10"] = aggregated_df["recall@10"] * query_factor
aggregated_df["penalized_ndcg@10"] = aggregated_df["ndcg@10"] * query_factor
aggregated_df

## 12. Create Per-Query Results Tables

Collect query-level ranking outputs and metrics in a DataFrame for inspection and export.

In [ ]:
per_query_df = pd.DataFrame(metrics["per_query"])
per_query_df["relevant_ids"] = [row["relevant_ids"] for row in filtered_benchmark]
per_query_df["hits@10"] = [binary_hits(pred, rel) for pred, rel in zip(per_query_df["retrieved_ids"], per_query_df["relevant_ids"])]
per_query_df.head()

## 13. Export Summary and Per-Query Artifacts

The full cloud runner writes all requested artifacts and returns DataFrames that can be displayed directly in the notebook.

In [ ]:
results = run_cloud_evaluation(
    data_folder=data_folder,
    benchmark_dir=Path("eval"),
    k=20,
    output_dir=Path("outputs/cloud_eval_notebook"),
    include_per_query=True,
)

results["summary"].head()

In [ ]:
results["manifest"]

In [ ]:
for model_state in fitted_retrievers.values():
    del model_state["retriever"]

gc.collect()